In [ ]:
from azure.storage.blob import BlobServiceClient
from io import BytesIO
import pandas as pd
import os
from dotenv import load_dotenv

In [2]:
def blob_to_df(container_client, blob_name):
    client = container_client.get_blob_client(blob_name)
    data = client.download_blob().readall()
    stream = BytesIO(data)
    df = pd.read_csv(stream)
    return df

In [3]:
load_dotenv()

True

In [5]:
connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
container_name = "recsys"

blob_service_client = BlobServiceClient.from_connection_string(connection_string)
container_client = blob_service_client.get_container_client(container_name)

In [ ]:
user_map_df = pd.DataFrame({
    "name": ['Qiqi','Torid', 'Uche'],
    'user_id': ['U19519', 'U12188', 'U22123']
})
user_map_df

In [ ]:
user_map = "user_map/user_map.csv"

container_client.get_blob_client(user_map).upload_blob(
        user_map_df.to_csv(index=False), overwrite=True)

In [6]:
news = pd.read_csv('news.tsv', header=None, sep='\t') #news.tsv is deleted and just uploaded from local
news.columns = [
    'content_id',
    "category",
    "sub_category",
    "title",
    "abstract",
    "URL",
    "title_entities",
    "abstract_entities"]

news = news.drop(columns=['URL', 'title_entities', 'abstract_entities'])
news = news.dropna()

In [ ]:
import re
import wordninja

def format_category(s):
    s = wordninja.split(s)
    s = " ".join(s)
    s = s.title()
    
    if s == 'Foodanddrink':
        s = 'Food and Drink'
    return s

def format_subcategory(s):
    s = wordninja.split(s)
    s = " ".join(s)
    s = s.title()

    if s == 'Lifestyle Ce Leb Style':
        s = 'Lifestyle Celeb Style'
    if s == 'Lifestyle Diy':
        s = 'DIY'

    return s

In [ ]:
news['category'] = news['category'].apply(format_category)
news['sub_category'] = news['sub_category'].apply(format_subcategory)

In [ ]:
raw_content = 'content/content.csv'

container_client.get_blob_client(raw_content).upload_blob(
        news.to_csv(index=False), overwrite=True)